# 04: Model D — Gemma 4-bit QLoRA Fine-Tuning with Robust T4 Fallback
**Course**: Foundations of Generative AI (ICT4403) - Assignment 3  
**Target Model**: `google/gemma-3-1b-it` (Fallback: `google/gemma-2-2b-it`)  
**Adaptation**: 4-bit NF4 QLoRA with rank $r=16$, $\alpha=32$ on all linear projections  
**Hardware Target**: Google Colab T4 GPU

### CRITICAL T4 VIVA POINT: Eager Attention and Numerical Fallback
> **Why `attn_implementation="eager"`?**  
> NVIDIA T4 GPUs lack FlashAttention-2 hardware support and lack bfloat16 matrix multiplication units. Gemma models frequently trigger numerical instabilities (fp16 overflow/underflow, NaN loss, or empty outputs) when running with experimental fused kernels or sliding window attention on T4. We enforce `attn_implementation="eager"` and implement an automated validation probe: if `google/gemma-3-1b-it` produces NaN loss or empty generation strings, the pipeline automatically falls back to `google/gemma-2-2b-it` with detailed logging.

## 1. Environment Setup, Colab Secrets & Drive Mount

In [ ]:
import os
import sys

DRIVE_DIR = "/content/drive/MyDrive/mt-fgai-assignment3"

try:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Google Drive ready at: {DRIVE_DIR}")
except ImportError:
    print("Running locally.")
    DRIVE_DIR = "."

if os.path.exists(DRIVE_DIR) and os.path.exists(os.path.join(DRIVE_DIR, "configs", "config.yaml")):
    REPO_ROOT = DRIVE_DIR
    os.chdir(REPO_ROOT)
    print(f"Active project root set to Google Drive: {REPO_ROOT}")
else:
    REPO_ROOT = os.path.abspath(".")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

try:
    import google.colab
    !pip install -q transformers>=4.45.0 accelerate peft trl bitsandbytes sentencepiece sacrebleu evaluate datasets pyyaml
except ImportError:
    pass

# Read Hugging Face token from Colab Secrets
hf_token = os.environ.get("HF_TOKEN", None)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    print("Successfully authenticated using Colab Secret HF_TOKEN.")
except Exception:
    if hf_token:
        print("Using HF_TOKEN from environment variables.")
    else:
        print("Warning: HF_TOKEN not found. If downloading gated models, ensure authentication.")

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


Mounting Google Drive...
Mounted at /content/drive
Google Drive ready at: /content/drive/MyDrive/mt-fgai-assignment3
Active project root set to Google Drive: /content/drive/MyDrive/mt-fgai-assignment3
Successfully authenticated using Colab Secret HF_TOKEN.
Active Device: cuda (Tesla T4)


## 2. Load Configuration and Splits

In [ ]:
import yaml
from src.data import load_split_from_jsonl

with open(os.path.join(REPO_ROOT, "configs", "config.yaml"), "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

splits_dir = os.path.join(REPO_ROOT, config["paths"]["splits_dir"])
train_data = load_split_from_jsonl("train", splits_dir=splits_dir)
val_data = load_split_from_jsonl("val", splits_dir=splits_dir)
test_data = load_split_from_jsonl("test", splits_dir=splits_dir)

subset_size = config["splits"]["llm_subset_size"]
train_subset = train_data[:subset_size]
print(f"Selected {len(train_subset):,} pairs for Gemma QLoRA fine-tuning.")

Selected 20,000 pairs for Gemma QLoRA fine-tuning.


## 3. Model Loading with Automated Fallback Protocol
Attempts to load `google/gemma-3-1b-it`. If numerical underflow or errors occur, falls back seamlessly to `google/gemma-2-2b-it`.

In [ ]:
import math
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from src.metrics import count_parameters

primary_id = config["models"]["gemma"]["primary_name"]
fallback_id = config["models"]["gemma"]["fallback_name"]
attn_impl = config["models"]["gemma"]["attn_implementation"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def try_load_gemma(target_id):
    print(f"Attempting to load {target_id} with attn_implementation='{attn_impl}'...")
    tok = AutoTokenizer.from_pretrained(target_id, token=hf_token)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    mdl = AutoModelForCausalLM.from_pretrained(
        target_id,
        quantization_config=bnb_config,
        device_map="auto",
        token=hf_token,
        attn_implementation=attn_impl,
    )

    # Quick numerical probe: compute forward pass on a 2-word prompt to test for NaN loss
    dummy_inputs = tok(["Hello world"], return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl(**dummy_inputs)
        if torch.isnan(out.logits).any():
            raise ValueError(f"NaN detected in logits of {target_id}")
    print(f"Successfully loaded and verified {target_id}!")
    return tok, mdl, target_id

selected_model_id = primary_id
try:
    tokenizer, model, active_model_id = try_load_gemma(primary_id)
except Exception as e:
    print(f"\n[WARNING] Primary model {primary_id} failed verification: {e}")
    print(f"Triggering fallback to: {fallback_id}")
    tokenizer, model, active_model_id = try_load_gemma(fallback_id)

base_params = count_parameters(model)
print(f"\nActive Gemma Model: {active_model_id}")
print(f"Total Parameters: {base_params['total_params']:,}")
print(f"Trainable Parameters: {base_params['trainable_params']:,}")

Attempting to load google/gemma-3-1b-it with attn_implementation='eager'...


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Successfully loaded and verified google/gemma-3-1b-it!

Active Gemma Model: google/gemma-3-1b-it
Total Parameters: 651,005,056
Trainable Parameters: 302,124,160


## 4. Zero-Shot Baseline Evaluation (1,000 Test Sentences)

In [ ]:
import os
import time
import torch
import sacrebleu
from tqdm import tqdm
from src.prompts import format_prompt_for_inference
from src.generation import clean_generated_output
from src.metrics import save_predictions_csv, save_metrics_json

zeroshot_metric_path = "results/metrics/metrics_gemma_zeroshot.json"
zeroshot_pred_path = "results/predictions/predictions_gemma_zeroshot.csv"

test_prompts = [format_prompt_for_inference(x["en"], tokenizer) for x in test_data]
all_references = [x["hi"] for x in test_data]
all_sources = [x["en"] for x in test_data]
all_ids = [x["id"] for x in test_data]

# 1. Register ALL valid Gemma stop tokens (CRITICAL: stops immediately after translation)
stop_token_ids = [tokenizer.eos_token_id]
for token_str in ["<end_of_turn>", "<eos>", "<|eot_id|>"]:
    t_id = tokenizer.convert_tokens_to_ids(token_str)
    if t_id is not None and t_id != tokenizer.unk_token_id and t_id not in stop_token_ids:
        stop_token_ids.append(t_id)

print(f"Active stop token IDs: {stop_token_ids}")
print(f"Evaluating {active_model_id} zero-shot on 1,000 sentences (batch_size=32, max_new_tokens=64)...")

# 2. Fast Batched Generation
model.eval()
device = next(model.parameters()).device
tokenizer.padding_side = "left"

batch_size = 32
max_new_tokens = 64
all_hypotheses = []
cleaned_count = 0
start_time = time.perf_counter()

for i in tqdm(range(0, len(test_prompts), batch_size), desc="Zero-Shot Batches"):
    batch_prompts = test_prompts[i : i + batch_size]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Greedy
            eos_token_id=stop_token_ids,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
    for raw in decoded:
        cleaned, was_mod = clean_generated_output(raw)
        all_hypotheses.append(cleaned)
        if was_mod:
            cleaned_count += 1

zs_time = time.perf_counter() - start_time
print(f"\nGeneration completed in {zs_time:.1f}s ({zs_time / len(test_data) * 1000.0:.2f} ms/sentence)!")

# 3. Compute Metrics
bleu_obj = sacrebleu.corpus_bleu(all_hypotheses, [all_references], tokenize="13a")
zs_bleu = round(float(bleu_obj.score), 2)
zs_bleu_sig = str(getattr(bleu_obj, "signature", "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp"))

chrf_obj = sacrebleu.corpus_chrf(all_hypotheses, [all_references])
chrf_pp_obj = sacrebleu.corpus_chrf(all_hypotheses, [all_references], word_order=2)
zs_chrf = round(float(chrf_obj.score), 2)
zs_chrf_pp = round(float(chrf_pp_obj.score), 2)
zs_chrf_sig = str(getattr(chrf_obj, "signature", "nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no"))

print(f"Gemma Zero-shot SacreBLEU: {zs_bleu} | chrF: {zs_chrf} | chrF++: {zs_chrf_pp}")

# 4. Save Artifacts
save_predictions_csv(all_ids, all_sources, all_references, all_hypotheses, zeroshot_pred_path)
save_metrics_json(
    model_name=f"{active_model_id}_zeroshot",
    architecture_type="decoder_only",
    adaptation_method="zero_shot",
    total_params=base_params["total_params"],
    trainable_params=0,
    train_data_size=0,
    train_time_sec=0.0,
    inference_time_total_sec=zs_time,
    num_test_samples=len(test_data),
    bleu=zs_bleu,
    bleu_signature=zs_bleu_sig,
    chrf=zs_chrf,
    chrf_pp=zs_chrf_pp,
    chrf_signature=zs_chrf_sig,
    decoding_settings={"strategy": "greedy", "max_new_tokens": max_new_tokens},
    batch_size=batch_size,
    hyperparameters={"model": active_model_id, "attention": attn_impl},
    output_path=zeroshot_metric_path,
    cleaned_outputs_count=cleaned_count,
)

Active stop token IDs: [1, 106]
Evaluating google/gemma-3-1b-it zero-shot on 1,000 sentences (batch_size=32, max_new_tokens=64)...


Zero-Shot Batches: 100%|██████████| 32/32 [03:18<00:00,  6.20s/it]



Generation completed in 198.6s (198.56 ms/sentence)!
Gemma Zero-shot SacreBLEU: 9.53 | chrF: 33.06 | chrF++: 31.18


{'model_name': 'google/gemma-3-1b-it_zeroshot',
 'architecture_type': 'decoder_only',
 'adaptation_method': 'zero_shot',
 'total_params': 651005056,
 'trainable_params': 0,
 'train_data_size': 0,
 'train_time_sec': 0.0,
 'inference_time_total_sec': 198.558,
 'inference_time_per_sentence_sec': 0.198558,
 'inference_time_per_sentence_ms': 198.56,
 'num_test_samples': 1000,
 'bleu': 9.53,
 'bleu_signature': 'nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp',
 'chrf': 33.06,
 'chrf_pp': 31.18,
 'chrf_signature': 'nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no',
 'batch_size': 32,
 'decoding_settings': {'strategy': 'greedy', 'max_new_tokens': 64},
 'cleaned_outputs_count': 97,
 'gpu': 'Tesla T4',
 'hyperparameters': {'model': 'google/gemma-3-1b-it', 'attention': 'eager'}}

## 5. Configure Gemma QLoRA Adapters & Response Masking

In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from src.prompts import ResponseOnlyDataCollator

model = prepare_model_for_kbit_training(model)
gemma_cfg = config["models"]["gemma"]

peft_config = LoraConfig(
    r=gemma_cfg["lora_r"],
    lora_alpha=gemma_cfg["lora_alpha"],
    lora_dropout=gemma_cfg["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=gemma_cfg["target_modules"],
)

peft_model = get_peft_model(model, peft_config)
peft_params = count_parameters(peft_model)
print(f"QLoRA Model Total Parameters: {peft_params['total_params']:,}")
print(f"QLoRA Trainable Adapter Parameters: {peft_params['trainable_params']:,} ({peft_params['trainable_params']/peft_params['total_params']*100:.2f}% of total)")

QLoRA Model Total Parameters: 664,050,816
QLoRA Trainable Adapter Parameters: 13,045,760 (1.96% of total)


## 6. Resumable Fine-Tuning Loop (Drive Checkpointing & Cumulative Timer)

In [ ]:
import json
import os
from datasets import Dataset
from transformers import Trainer, TrainingArguments, TrainerCallback, DataCollatorForSeq2Seq
from src.metrics import CumulativeTimer

train_subset = train_data[:2500]

output_dir = os.path.join(DRIVE_DIR, "gemma_qlora_checkpoints")
os.makedirs(output_dir, exist_ok=True)
timer_file = os.path.join(output_dir, "cumulative_time.json")

init_time = 0.0
if os.path.exists(timer_file):
    with open(timer_file, "r") as f:
        init_time = json.load(f).get("cumulative_seconds", 0.0)
    print(f"Restored previous cumulative training time: {init_time:.1f}s")

timer = CumulativeTimer(initial_seconds=init_time)
timer.start()

class CumulativeTimeCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        with open(timer_file, "w") as f:
            json.dump({"cumulative_seconds": timer.elapsed(), "step": state.global_step}, f)

# 1. Pre-tokenize the training subset with response-only masking (-100 on prompts)
print(f"Preparing {len(train_subset):,} pairs with prompt masking (-100)...")
formatted_data = []
for item in train_subset:
    src = item["en"]
    tgt = item["hi"]
    prompt_msgs = [{"role": "user", "content": f"Translate the following English sentence to Hindi. Output only the Hindi translation.\n\nEnglish: {src}"}]
    prompt_str = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    prompt_ids = tokenizer.encode(prompt_str, add_special_tokens=False)

    full_msgs = [{"role": "user", "content": f"Translate the following English sentence to Hindi. Output only the Hindi translation.\n\nEnglish: {src}"}, {"role": "assistant", "content": tgt}]
    full_str = tokenizer.apply_chat_template(full_msgs, tokenize=False, add_generation_prompt=False)
    full_ids = tokenizer.encode(full_str, add_special_tokens=False)
    if not full_ids or full_ids[-1] != tokenizer.eos_token_id:
        full_ids.append(tokenizer.eos_token_id)

    full_ids = full_ids[:256]
    prompt_len = min(len(prompt_ids), len(full_ids))
    labels = [-100] * prompt_len + full_ids[prompt_len:]
    formatted_data.append({
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    })

train_dataset = Dataset.from_list(formatted_data)

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="no",
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    learning_rate=gemma_cfg["lr"],
    per_device_train_batch_size=gemma_cfg["batch_size"],
    gradient_accumulation_steps=gemma_cfg["gradient_accumulation_steps"],
    num_train_epochs=gemma_cfg["epochs"],
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=25,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    callbacks=[CumulativeTimeCallback()],
)

has_checkpoint = any(d.startswith("checkpoint-") for d in os.listdir(output_dir))
print(f"Resume from checkpoint available: {has_checkpoint}")
print(f"Training on {len(train_dataset):,} pairs (~{len(train_dataset) // (gemma_cfg['batch_size'] * gemma_cfg['gradient_accumulation_steps'])} gradient steps)...")

trainer.train(resume_from_checkpoint=has_checkpoint)
total_train_time = timer.stop()

# Save final LoRA adapter to Google Drive
adapter_dir = os.path.join(DRIVE_DIR, "lora_gemma_hindi")
peft_model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
with open(timer_file, "w") as f:
    json.dump({"cumulative_seconds": total_train_time}, f)
print(f"\nQLoRA training complete! Total time: {total_train_time:.1f}s. Adapter saved to: {adapter_dir}")

Preparing 2,500 pairs with prompt masking (-100)...
Resume from checkpoint available: False
Training on 2,500 pairs (~78 gradient steps)...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
25,2.826770
50,2.691150
75,2.557980



QLoRA training complete! Total time: 353.3s. Adapter saved to: /content/drive/MyDrive/mt-fgai-assignment3/lora_gemma_hindi


## 7. Fine-Tuned Gemma Test Evaluation (1,000 Sentences)
Evaluates the adapted Gemma model using left-padded greedy decoding on the identical test set.

In [ ]:
import sacrebleu
from src.generation import generate_hf_causal_lm
from src.metrics import save_predictions_csv, save_metrics_json

print(f"Evaluating fine-tuned {active_model_id} on 1,000 test sentences...")
hypotheses, inf_time, cleaned_count = generate_hf_causal_lm(
    model=peft_model,
    tokenizer=tokenizer,
    prompts=test_prompts,
    batch_size=16,
    max_new_tokens=config["generation"]["max_new_tokens"],
    temperature=0.0,
)

bleu_obj = sacrebleu.corpus_bleu(hypotheses, [all_references], tokenize="13a")
bleu_score = round(float(bleu_obj.score), 2)
bleu_sig = str(getattr(bleu_obj, "signature", "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp"))

chrf_obj = sacrebleu.corpus_chrf(hypotheses, [all_references])
chrf_pp_obj = sacrebleu.corpus_chrf(hypotheses, [all_references], word_order=2)
chrf_score = round(float(chrf_obj.score), 2)
chrf_pp_score = round(float(chrf_pp_obj.score), 2)
chrf_sig = str(getattr(chrf_obj, "signature", "nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no"))

print(f"\n=== Model D ({active_model_id} QLoRA) Test Evaluation ===")
print(f"SacreBLEU: {bleu_score} (Signature: {bleu_sig})")
print(f"chrF     : {chrf_score}")
print(f"chrF++   : {chrf_pp_score}")
print(f"Inference Latency: {inf_time / len(test_data) * 1000.0:.2f} ms/sentence")
print(f"Cleaned Outputs Count: {cleaned_count}")

pred_path = "results/predictions/predictions_gemma_qlora.csv"
metric_path = "results/metrics/metrics_gemma_qlora.json"

save_predictions_csv(all_ids, all_sources, all_references, hypotheses, pred_path)
save_metrics_json(
    model_name=f"{active_model_id}_qlora",
    architecture_type="decoder_only",
    adaptation_method="qlora_4bit",
    total_params=peft_params["total_params"],
    trainable_params=peft_params["trainable_params"],
    train_data_size=len(train_subset),
    train_time_sec=total_train_time,
    inference_time_total_sec=inf_time,
    num_test_samples=len(test_data),
    bleu=bleu_score,
    bleu_signature=bleu_sig,
    chrf=chrf_score,
    chrf_pp=chrf_pp_score,
    chrf_signature=chrf_sig,
    decoding_settings={"strategy": "greedy", "max_new_tokens": config["generation"]["max_new_tokens"]},
    batch_size=16,
    hyperparameters={"model": active_model_id, "attention": attn_impl, **gemma_cfg},
    output_path=metric_path,
    cleaned_outputs_count=cleaned_count,
)

print("\n--- Sample Test Predictions ---")
for i in range(5):
    print(f"\n[Test Sample {i+1}]")
    print(f"  Source    : {all_sources[i]}")
    print(f"  Reference : {all_references[i]}")
    print(f"  Hypothesis: {hypotheses[i]}")

Evaluating fine-tuned google/gemma-3-1b-it on 1,000 test sentences...

=== Model D (google/gemma-3-1b-it QLoRA) Test Evaluation ===
SacreBLEU: 11.1 (Signature: nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp)
chrF     : 35.93
chrF++   : 34.06
Inference Latency: 454.28 ms/sentence
Cleaned Outputs Count: 0

--- Sample Test Predictions ---

[Test Sample 1]
  Source    : The US warns its own companies about economic espionage by other countries.
  Reference : अमेरिका ने अन्य देशों द्वारा अपनी ही कंपनियों की आर्थिक जासूसी के बारे में चेतावनी दी है।
  Hypothesis: यू.एस. अपने देश के व्यवसायों को आर्थिक जासूसी के बारे में चेतावनी देता है।

[Test Sample 2]
  Source    : Most of the skirmishes have taken place in Sofala province, which is several hundred kilometres north of Maputo, but hosts Beira, the port that miners, including Rio Tinto and Vale, rely on to export coal.
  Reference : झड़पों के अधिकांशसोफा ला प्रांत मेंहै, जोमापुटो के उत्तर में कई सौ किलोमीटर की दूरी पर है, जिसमें मेजबान बीरा पोर